# UK Biobank (UKB) Data Processing Pipeline

This notebook implements a comprehensive data processing pipeline for UK Biobank data, focusing on participant selection and subgroup classification. The pipeline is designed to create well-defined groups for subsequent normative modeling and analysis.

## Overview

The pipeline consists of several key steps:

1. **Initial Data Loading**
   - Loads raw UKB data
   - Extracts relevant fields for participant selection
   - Filters for participants with available MRI data

2. **Healthy Control (HC) Group Creation**
   - Applies systematic exclusion criteria:
     - ICD-10 diagnostic codes
     - OPCS-4 procedure codes
     - Self-reported illness codes
     - Questionnaire responses
   - Creates a clean HC reference dataset

3. **IBS Participant Identification**
   - Identifies participants with Irritable Bowel Syndrome
   - Creates three distinct subgroups:
     - Group 1: ROME III criteria
     - Group 2: Clinical diagnosis
     - Group 3: Overlap group

4. **MRI Data Integration**
   - Ensures MRI data availability
   - Extracts key covariates:
     - Age
     - Sex
     - BMI
     - Imaging site
   - Applies age filter (>44 years)

5. **Final Dataset Preparation**
   - Creates cleaned, documented datasets
   - Saves outputs for downstream analysis

## Usage Instructions

1. Set the correct paths in the first code cell:
   - `raw_data_dir`: Path to raw UKB data
   - `root_dir`: Path to project root
   - `out_dir`: Path for output files
   - `docu_dir`: Path to documentation files

2. Run cells sequentially to:
   - Load and process raw data
   - Apply exclusion criteria
   - Create final datasets

3. Output files will be saved in the specified output directory

## Output Files

The pipeline generates the following key output files:
- `HC_reference_MRI_age.csv`: Reference healthy control dataset
- `HC_MRI_age.csv`: Healthy control dataset
- `IBS_Both_MRI_age.csv`: IBS participants meeting both criteria
- `IBS_ROMEIII_MRI_age.csv`: IBS participants by ROME III criteria
- `IBS_Diagnosed_MRI_age.csv`: Clinically diagnosed IBS participants

In [1]:
import os
import pandas as pd

raw_data_dir = 'Absolute data path'
rawdata_csv = os.path.join(raw_data_dir, 'ukb52200.csv')
MRIdata_csv = os.path.join(raw_data_dir, 'ukb678714.csv')

root_dir = 'Abosolute path to this project'
out_dir = os.path.join(root_dir, '3_rerun_whole_work', '1_data_cleaned')
docu_dir = os.path.join(root_dir, '1_document')
os.makedirs(out_dir,exist_ok=True)

# Related covariates for data exclusion, N = 53330 left

In [ ]:
# 21862-2.0 Date visit Assessment center, 41270 ICD-10
usecols = (['eid', '31-0.0', '21862-2.0'] + ['41270-0.'+str(icd_array) for icd_array in range(0, 243)] + 
           # 41272 OPCS4
           ['41272-0.'+str(icd_array) for icd_array in range(0, 124)] + 
           # DHQ
           ['21025-0.0', '21026-0.0', '21027-0.0', '21028-0.0', '21029-0.0', '21030-0.0', '21031-0.0', 
            '21032-0.0', '21033-0.0', '21034-0.0', '21069-0.0', '21024-0.0', '21069-0.0'] + 
            # 20002 self-report illness
           ['20002-0.'+str(icd_array) for icd_array in range(0, 34)] + 
           ['20002-1.'+str(icd_array) for icd_array in range(0, 34)] + 
           ['20002-2.'+str(icd_array) for icd_array in range(0, 34)] + 
           ['20002-3.'+str(icd_array) for icd_array in range(0, 34)] +
           # Mental health
           ['2050-2.0', '2060-2.0', '4526-2.0', '1930-2.0', '1960-2.0', '1170-2.0', '1200-2.0', 
            '2070-2.0', '2080-2.0', '1940-2.0', '1950-2.0', '2020-2.0', '2030-2.0', '20446-0.0', 
            '20441-0.0', '20450-0.0', '20449-0.0', '20435-0.0', '20427-0.0', '20536-0.0', '20532-0.0', 
            '20439-0.0', '20436-0.0', '20440-0.0']
           )

All_data_chunk = pd.read_csv(rawdata_csv, chunksize=1000, usecols=usecols)
All_data_with_MRI_visit = pd.DataFrame()
for chunk in All_data_chunk:
    # Drop rows where column '21862-2.0' has NaN (drop subjects without MRI data assessment)
    chunk.dropna(subset=['21862-2.0'], inplace=True)
    All_data_with_MRI_visit = pd.concat([All_data_with_MRI_visit, chunk], axis=0)
All_data_with_MRI_visit.dropna(how='all',axis=1,inplace=True)
All_data_with_MRI_visit.to_csv(os.path.join(out_dir, 'brain_mri_signoff.csv'),index=False)

# HC inclusion

## Exclusion with ICD-10, N = 32154 left

In [ ]:
# Load ICD-10 codes to exclude
with open(os.path.join(docu_dir, 'icd_10_HC.txt'), 'r') as filestream:
    HC_ex_icd_10 = filestream.read().split(", ")
ex_rules_hc = '|'.join(HC_ex_icd_10)

HC_data_icd_10 = pd.read_csv(os.path.join(out_dir, 'brain_mri_signoff.csv'))
columns_icd10 = HC_data_icd_10.filter(like='41270').columns.tolist()
mask = ~HC_data_icd_10[columns_icd10].apply(lambda col: col.str.contains(ex_rules_hc, na=False)).any(axis=1)
HC_data_icd_10 = HC_data_icd_10[mask].reset_index(drop=True)

HC_data_icd_10.dropna(how='all', axis=1, inplace=True)
HC_data_icd_10.to_csv(os.path.join(out_dir, 'HC_s1_cleaned_icd.csv'), index=False)

## Exclusion with OPCS4, N = 32120 left

In [ ]:
with open(os.path.join(docu_dir, 'OPCS4_HC.txt'), 'r') as filestream:
    HC_ex_opcs4 = filestream.read().split(", ")
ex_rules_hc = '|'.join(HC_ex_opcs4)

HC_data_opcs4 = pd.read_csv(os.path.join(out_dir, 'HC_s1_cleaned_icd.csv'))
columns_opcs4 = HC_data_opcs4.filter(like='41272').columns.tolist()
mask = ~HC_data_opcs4[columns_opcs4].apply(lambda col: col.str.contains(ex_rules_hc, na=False)).any(axis=1)
HC_data_opcs4 = HC_data_opcs4[mask].reset_index(drop=True)

HC_data_opcs4.dropna(how='all', axis=1, inplace=True)
HC_data_opcs4.to_csv(os.path.join(out_dir, 'HC_s2_cleaned_OPCS4.csv'), index=False)

## Exclusion with DHQ, N = 21947 left

In [ ]:
HC_data_dhq = pd.read_csv(os.path.join(out_dir, 'HC_s2_cleaned_OPCS4.csv'))

conditions = (
    (~HC_data_dhq['21025-0.0'].isin(range(2, 7))) &
    (~HC_data_dhq['21033-0.0'].isin(range(-504, -501))) &
    (~HC_data_dhq['21034-0.0'].isin(range(-504, -501))) &
    (~HC_data_dhq['21069-0.0'].isin(range(-705, -702))) &
    (HC_data_dhq['21024-0.0'] != 1)
)

HC_data_dhq = HC_data_dhq[conditions].reset_index(drop=True)
HC_data_dhq.dropna(how='all',axis=1,inplace=True)
HC_data_dhq.to_csv(os.path.join(out_dir, 'HC_s3_cleaned_DHQ.csv'), index=False)

## Exclusion with self-report illness, N = 21166 left

In [ ]:
HC_data_sr = pd.read_csv(os.path.join(out_dir, 'HC_s3_cleaned_DHQ.csv'))

# Illness codes to exclude
SR_num = {1135, 1154, 1164, 1165, 1191, 1456, 1458, 1459, 1461,
          1462, 1463, 1509, 1510, 1562, 1599, 1600, 1601, 1602}

columns_sr = HC_data_sr.filter(like='20002').columns.tolist()
HC_data_sr = HC_data_sr[~HC_data_sr[columns_sr].isin(SR_num).any(axis=1)].reset_index(drop=True)
HC_data_sr.dropna(how='all', axis=1, inplace=True)
HC_data_sr.to_csv(os.path.join(out_dir, 'HC_s4_cleaned_sr.csv'), index=False)

# IBS dataset clean

## Exclusion with coeliac/gluten sensitivity, N = 52972 left

In [ ]:
IBS_gluten = pd.read_csv(os.path.join(out_dir, 'brain_mri_signoff.csv'))
IBS_gluten = IBS_gluten[~IBS_gluten['21069-0.0'].isin(range(-705, -702))].reset_index(drop=True)
IBS_gluten.dropna(how='all', axis=1, inplace=True)
IBS_gluten.to_csv(os.path.join(out_dir, 'IBS_s1_cleaned_celiac.csv'), index=False)

## Exclusion with ICD-10, N = 35590 left

In [ ]:
with open(os.path.join(docu_dir, 'icd_10_IBS.txt'), 'r') as filestream:
    IBS_ex_icd_10 = filestream.read().split(", ")
ex_rules_ibs = '|'.join(IBS_ex_icd_10)

IBS_icd10 = pd.read_csv(os.path.join(out_dir, 'IBS_s1_cleaned_celiac.csv'))
columns_icd10 = IBS_icd10.filter(like='41270').columns.tolist()
mask = ~IBS_icd10[columns_icd10].apply(lambda col: col.str.contains(ex_rules_ibs, na=False)).any(axis=1)

IBS_icd10 = IBS_icd10[mask].reset_index(drop=True)
IBS_icd10.dropna(how='all', axis=1, inplace=True)
IBS_icd10.to_csv(os.path.join(out_dir, 'IBS_s2_cleaned_icd.csv'), index=False)

## Exclusion with OPCS4, N = 35535 left

In [ ]:
with open(os.path.join(docu_dir, 'OPCS4_IBS.txt'), 'r') as filestream:
    IBS_ex_opcs4 = filestream.read().split(", ")
ex_rules_ibs = '|'.join(IBS_ex_opcs4)

IBS_opcs4 = pd.read_csv(os.path.join(out_dir, 'IBS_s2_cleaned_icd.csv'))
columns_opcs4 = IBS_opcs4.filter(like='41272').columns.tolist()
mask = ~IBS_opcs4[columns_opcs4].apply(lambda col: col.str.contains(ex_rules_ibs, na=False)).any(axis=1)

IBS_opcs4 = IBS_opcs4[mask].reset_index(drop=True)
IBS_opcs4.dropna(how='all', axis=1, inplace=True)
IBS_opcs4.to_csv(os.path.join(out_dir, 'IBS_s3_cleaned_OPCS4.csv'), index=False)

## Exclusion with self-report illness, N = 35088 left

In [ ]:
IBS_sr = pd.read_csv(os.path.join(out_dir, 'IBS_s3_cleaned_OPCS4.csv'))

# Illness codes to exclude
SR_num = {1135, 1164, 1165, 1191, 1456, 1459, 1461, 1462, 1463, 1509, 1600, 1601, 1602}

columns_sr = IBS_sr.filter(like='20002').columns.tolist()
IBS_sr = IBS_sr[~IBS_sr[columns_sr].isin(SR_num).any(axis=1)].reset_index(drop=True)
IBS_sr.dropna(how='all', axis=1, inplace=True)
IBS_sr.to_csv(os.path.join(out_dir, 'IBS_s4_cleaned_sr.csv'), index=False)

# IBS inclusion with multiple criteria

## IBS with ROME III

In [ ]:
IBS_cleaned = pd.read_csv(os.path.join(out_dir, 'IBS_s4_cleaned_sr.csv'))

IBS_DHQ_ROME = IBS_cleaned[
    (IBS_cleaned['21025-0.0'] > 2) &
    ((IBS_cleaned['21026-0.0'] < 1) | (IBS_cleaned['31-0.0'] == 1)) &
    (IBS_cleaned['21027-0.0'] == 1)
].reset_index(drop=True)

# Define range conditions clearly and concisely
def condition_between(series, lower, upper):
    return (series > lower) & (series < upper)

# Create simplified, clearly named conditions
c1 = condition_between(IBS_DHQ_ROME['21028-0.0'], -600, -500)
c2 = (
    condition_between(IBS_DHQ_ROME['21029-0.0'], -600, -500) |
    condition_between(IBS_DHQ_ROME['21030-0.0'], -600, -500)
)
c3 = (
    condition_between(IBS_DHQ_ROME['21031-0.0'], -600, -500) |
    condition_between(IBS_DHQ_ROME['21032-0.0'], -600, -500)
)

# Final combined filtering
IBS_DHQ_ROME = IBS_DHQ_ROME[(c1 & c2) | (c1 & c3) | (c2 & c3)].reset_index(drop=True)

## IBS self-report in DHQ

In [ ]:
IBS_cleaned = pd.read_csv(os.path.join(out_dir, 'IBS_s4_cleaned_sr.csv'))
IBS_DHQ_SR = IBS_cleaned.loc[IBS_cleaned['21024-0.0']==1]

## IBS Unprompted self-report

In [ ]:
IBS_cleaned = pd.read_csv(os.path.join(out_dir, 'IBS_s4_cleaned_sr.csv'))
columns_sr = IBS_cleaned.filter(like='20002').columns.tolist()
IBS_SR = IBS_cleaned[(IBS_cleaned[columns_sr] == 1154).any(axis=1)]

## IBS Hospital ICD-10

In [ ]:
IBS_cleaned = pd.read_csv(os.path.join(out_dir, 'IBS_s4_cleaned_sr.csv'))
columns_sr = IBS_cleaned.filter(like='41270').columns.tolist()
IBS_ICD10 = IBS_cleaned[(IBS_cleaned[columns_sr] == 'K580').any(axis=1)|(IBS_cleaned[columns_sr] == 'K589').any(axis=1)]

## Convert to 3 IBS subgroups: ROME III, Diagnosed, Both

In [40]:
# Apply logic to assign final group
def assign_group(row):
    if row['_merge'] == 'left_only':
        return row['Group']
    elif row['_merge'] == 'both' and row['Group'] in [1, 3]:
        return 3
    elif row['_merge'] == 'both' and row['Group'] == 2:
        return 2
    elif row['_merge'] == 'right_only':
        return 2
    return None

In [ ]:
IBS_DHQ_ROME['Group'] = 1
IBS_DHQ_SR['Group'] = 2
IBS_SR['Group'] = 2
IBS_ICD10['Group'] = 2
IBS_all_groups = pd.merge(IBS_DHQ_ROME[['eid']], IBS_DHQ_SR[['eid']], how='outer', on=['eid'], indicator=True)
IBS_all_groups['Group'] = IBS_all_groups['_merge'].map({
    'left_only': 1,
    'right_only': 2,
    'both': 3
})
IBS_all_groups = pd.merge(IBS_all_groups[['eid', 'Group']], IBS_SR[['eid']], how='outer', on=['eid'], indicator=True)
IBS_all_groups['Group'] = IBS_all_groups.apply(assign_group, axis=1)
IBS_all_groups = pd.merge(IBS_all_groups[['eid', 'Group']], IBS_ICD10[['eid']], how='outer', on=['eid'], indicator=True)
IBS_all_groups['Group'] = IBS_all_groups.apply(assign_group, axis=1)

In [62]:
IBS_all_groups[['eid','Group']].to_csv(os.path.join(out_dir, 'IBS_All_idx.csv'), index=False)
# save 3 groups seperately
IBS_all_groups['eid'][IBS_all_groups['Group'] == 1].to_csv(os.path.join(out_dir, 'IBS_ROMEIII_idx.csv'), index=False)
IBS_all_groups['eid'][IBS_all_groups['Group'] == 2].to_csv(os.path.join(out_dir, 'IBS_Diagnosed_idx.csv'), index=False)
IBS_all_groups['eid'][IBS_all_groups['Group'] == 3].to_csv(os.path.join(out_dir, 'IBS_Both_idx.csv'), index=False)

# MDD exclused from HC dataset

In [ ]:
IBS_cleaned = pd.read_csv(os.path.join(out_dir, 'IBS_s4_cleaned_sr.csv'))
condition_a = IBS_cleaned['2050-2.0'].isin([3, 4])
condition_b = IBS_cleaned['2060-2.0'].isin([3, 4])
condition_c = IBS_cleaned['4526-2.0'].isin([5, 6])

condition_d_a = (
    (IBS_cleaned['1930-2.0'] == 1) |
    (IBS_cleaned['1960-2.0'] == 1) |
    (IBS_cleaned['2050-2.0'] > 1) |
    (IBS_cleaned['2060-2.0'] > 1) |
    (IBS_cleaned['4526-2.0'] > 3)
)

condition_d_b = (
    (IBS_cleaned['1170-2.0'] == 1) |
    (IBS_cleaned['1200-2.0'] == 3)
)

condition_d_c = (
    (IBS_cleaned['2070-2.0'] == 4) |
    (IBS_cleaned['2080-2.0'] == 4)
)

condition_d_d = (
    (IBS_cleaned['1940-2.0'] == 1) |
    (IBS_cleaned['1950-2.0'] == 1) |
    (IBS_cleaned['2020-2.0'] == 1) |
    (IBS_cleaned['2030-2.0'] == 1)
)

condition_d = (
    (condition_d_a & condition_d_b & condition_d_c) |
    (condition_d_a & condition_d_b & condition_d_d) |
    (condition_d_a & condition_d_c & condition_d_d) |
    (condition_d_b & condition_d_c & condition_d_d)
)

cMDD = IBS_cleaned[condition_a | condition_b | condition_c | condition_d][['eid']]

# --- Block 2: pMDD_CIDI conditions ---
condition_a_pMDD = (
    (IBS_cleaned['20446-0.0'] == 1).astype(int) +
    (IBS_cleaned['20441-0.0'] == 1).astype(int) +
    (IBS_cleaned['20450-0.0'] == 1).astype(int) +
    (IBS_cleaned['20449-0.0'] == 1).astype(int) +
    (IBS_cleaned['20435-0.0'] == 1).astype(int) +
    (IBS_cleaned['20427-0.0'] == 1).astype(int) +
    (IBS_cleaned['20536-0.0'] > 0).astype(int) +
    (IBS_cleaned['20532-0.0'] == 1).astype(int)
) > 4

condition_b_pMDD = (
    (IBS_cleaned['20439-0.0'] > 1) &
    (IBS_cleaned['20436-0.0'] > 2)
)

condition_c_pMDD = IBS_cleaned['20440-0.0'] > 0

pMDD_CIDI = IBS_cleaned[condition_a_pMDD & condition_b_pMDD & condition_c_pMDD][['eid']]

# --- Block 3: ICD-code-based conditions (F32) ---
icd_name = IBS_cleaned.filter(like='41270').columns.tolist()
mask_icd = IBS_cleaned[icd_name].apply(lambda col: col.str.startswith('F32', na=False))
MDD_ICD = IBS_cleaned.loc[mask_icd.any(axis=1), ['eid']]

# --- Merge all identified MDD patients together ---
MDD_all = pd.concat([cMDD, pMDD_CIDI, MDD_ICD]).drop_duplicates().reset_index(drop=True)

In [ ]:
HC_data = pd.read_csv(os.path.join(out_dir, 'HC_s4_cleaned_sr.csv'))
HC_data_cleaned = HC_data[~HC_data['eid'].isin(MDD_all['eid'])].reset_index(drop=True)
HC_data_cleaned = HC_data_cleaned[~HC_data_cleaned['eid'].isin(IBS_all_groups['eid'])].reset_index(drop=True)
HC_data_cleaned[['eid']].to_csv(os.path.join(out_dir, 'HC_cleaned_idx.csv'), index=False)

# MRI data inclusion

In [ ]:
cleaned_HC = pd.read_csv(os.path.join(out_dir, 'HC_cleaned_idx.csv'), usecols=['eid'])
IDP_name = ['eid'] + [str(icd_array)+'-2.0' for icd_array in range(27329, 27773)]
MRI_data_chunk = pd.read_csv(MRIdata_csv, chunksize=100000, usecols=IDP_name)
HC_MRI_data = pd.DataFrame()
for chunk in MRI_data_chunk:
    HC_MRI_data = pd.concat([HC_MRI_data, pd.merge(cleaned_HC, chunk, how='inner', on='eid')], axis=0)
HC_MRI_data.dropna(how='all',axis=1,inplace=True)
HC_MRI_data.reset_index(drop=True, inplace=True)
# Drop subjects with more than 10 idp lost (444 IDP in total)
HC_MRI_data.dropna(thresh=434, inplace=True)
HC_MRI_data.to_csv(os.path.join(out_dir, 'HC_MRI_all.csv'),index=False)

In [3]:
IBS_all = pd.read_csv(os.path.join(out_dir, 'IBS_All_idx.csv'))
IDP_name = ['eid'] + [str(icd_array)+'-2.0' for icd_array in range(27329, 27773)]
MRI_data_chunk = pd.read_csv(MRIdata_csv, chunksize=100000, usecols=IDP_name)
IBS_all_MRI_data = pd.DataFrame()
for chunk in MRI_data_chunk:
    IBS_all_MRI_data = pd.concat([IBS_all_MRI_data, pd.merge(IBS_all[['eid']], chunk, how='inner', on='eid')], axis=0)
IBS_all_MRI_data = IBS_all.merge(IBS_all_MRI_data, on='eid', how='left')
IBS_all_MRI_data.dropna(how='all',axis=1,inplace=True)
IBS_all_MRI_data.reset_index(drop=True, inplace=True)
# Drop subjects with more than 10 idp lost (444 IDP in total)
IBS_all_MRI_data.dropna(thresh=434, inplace=True)
IBS_all_MRI_data.to_csv(os.path.join(out_dir, 'IBS_All_MRI_all.csv'),index=False)

# Covariates inclusion

In [ ]:
datasets = {
    'HC': pd.read_csv(os.path.join(out_dir, 'HC_MRI_all.csv')),
    'IBS_All': pd.read_csv(os.path.join(out_dir, 'IBS_All_MRI_all.csv')),
}

cov_name = ['eid', '21003-2.0', '31-0.0', '21001-2.0', '54-2.0']
rawdata_chunks = pd.read_csv(rawdata_csv, chunksize=100000, usecols=cov_name)

cov_dfs = {name: pd.DataFrame() for name in datasets}

for chunk in rawdata_chunks:
    for name, df in datasets.items():
        merged_chunk = chunk.merge(df[['eid']], on='eid', how='inner')
        cov_dfs[name] = pd.concat([cov_dfs[name], merged_chunk], ignore_index=True)

for name, cov_df in cov_dfs.items():
    cov_df.dropna(how='all', axis=1, inplace=True)
    cov_df.dropna(inplace=True)
    cov_df = cov_df[cov_df['21003-2.0'] > 44].reset_index(drop=True)

    final_merged = datasets[name].merge(cov_df, on='eid', how='inner')
    out_file = os.path.join(out_dir, f'{name}_MRI_age.csv')
    print(f"{name}: {len(final_merged)}")
    final_merged.to_csv(out_file, index=False)

# HC spliation for HC reference dataset and HC group for further analyses with IBS (matched by PSM)

In [66]:
from psmpy import PsmPy

HC_data = pd.read_csv(os.path.join(out_dir, 'HC_MRI_age.csv'), usecols=['eid', '21003-2.0', '31-0.0', '54-2.0'])
HC_data.rename(columns={'21003-2.0':'age', '31-0.0':'sex', '54-2.0':'site'},inplace=True)

pat_data = pd.read_csv(os.path.join(out_dir, 'IBS_All_MRI_age.csv'), usecols=['eid', '21003-2.0', '31-0.0', '54-2.0'])
pat_data.rename(columns={'21003-2.0':'age', '31-0.0':'sex', '54-2.0':'site'},inplace=True)

HC_data = HC_data[['eid','age','sex','site']]
pat_data = pat_data[['eid','age','sex','site']]
HC_data['group_id'] = 0
pat_data['group_id'] = 1
HC_data = pd.get_dummies(HC_data,columns=['site'])
pat_data = pd.get_dummies(pat_data,columns=['site'])

In [ ]:
psm_group = PsmPy(pd.concat([HC_data,pat_data]), treatment='group_id', indx='eid', exclude = [])
psm_group.logistic_ps(balance = True)
psm_group.knn_matched(matcher='propensity_logit', replacement=False, caliper=None, drop_unmatched=True)
psm_group.matched_ids.to_csv(os.path.join(out_dir, 'IBS_matched_ids.csv'),index=False)
psm_group.df_matched.to_csv(os.path.join(out_dir, 'IBS_matched_id_propensity.csv'),index=False)
psm_group.effect_size_plot(title='Standardized Mean differences accross covariates before and after matching', before_color='#FCB754', after_color='#3EC8FB', save=False)
psm_group.effect_size.to_csv(os.path.join(out_dir, 'IBS_update_matched_ids_effectsize.csv'),index=False)

## Splite healthy data to reference dataset and case dataset

In [3]:
HC_eid_case = pd.read_csv(os.path.join(out_dir, 'IBS_matched_ids.csv'),usecols=['matched_ID'])
HC_data = pd.read_csv(os.path.join(out_dir, 'HC_MRI_age.csv'))
HC_reference = HC_data[~HC_data['eid'].isin(HC_eid_case['matched_ID'])]
HC = HC_data[HC_data['eid'].isin(HC_eid_case['matched_ID'])]
HC_reference.to_csv(os.path.join(out_dir, 'HC_reference_MRI_age.csv'),index=False)
HC.to_csv(os.path.join(out_dir, 'HC_MRI_age.csv'),index=False)